In [ ]:
import pandas as pd
import numpy as np 
import os
import json

In [ ]:
artifacts_path = "../exp_results/artifacts/"

# Create the folder if it doesn't exist
os.makedirs(artifacts_path, exist_ok=True)

existing_experiments = []
if os.path.exists(artifacts_path):
    for folder_name in os.listdir(artifacts_path):
        folder_path = os.path.join(artifacts_path, folder_name)
        if os.path.isdir(folder_path) and folder_name.isdigit():
            existing_experiments.append(int(folder_name))


EXP_NUMBER = max(existing_experiments, default=0) + 1

#create the experience folder
os.makedirs(f"{artifacts_path}/{EXP_NUMBER}")

NUM_DATA_POINTS = 50000

print(f"Current experiment number: {EXP_NUMBER}")

**load the data**

In [ ]:

#read the json file
data_file_path = "../data/processed/multinli_1.0_train_cleaned.csv"
data = pd.read_csv(data_file_path, sep='µ')

#get a random sample of the data
data = data.sample(n = NUM_DATA_POINTS, random_state = 42)
data.head()

### Apply TF-IDF

In [ ]:
import spacy
import re
import unicodedata
from tqdm import tqdm

nlp = spacy.load(
    "en_core_web_sm",
    disable=["parser", "ner"]  # disable unused components
)
nlp.max_length = 10_000_000  # safety for large inputs

def clean_text(text):
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8', 'ignore')
    text = re.sub(r"[^a-zA-Z\s]", ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

def preprocess_texts(texts, batch_size=500, n_process=-1):
    texts = [clean_text(t) for t in texts]
    
    cleaned = []
    for doc in tqdm(
        nlp.pipe(texts, batch_size=batch_size, n_process=n_process),
        total=len(texts)
    ):
        tokens = [token.lemma_ for token in doc]
        cleaned.append(" ".join(tokens))

    return cleaned

In [ ]:
#clean text sentence1 and sentence2
data['sentence1_cleaned'] = preprocess_texts(data['sentence1'].astype(str).tolist())
data['sentence2_cleaned'] = preprocess_texts(data['sentence2'].astype(str).tolist())

In [ ]:
#split before tf-idf to avoid leakage
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

data_train, data_val = train_test_split(
    data,
    test_size=0.1,
    random_state=42,
    stratify=data["gold_label"]
)

# Split train and validation text for TF-IDF fitting
all_sentences_train = data_train["sentence1_cleaned"].tolist() + data_train["sentence2_cleaned"].tolist()

tfidf = TfidfVectorizer(
    min_df=0.001,
    ngram_range=(1, 2)
)
tfidf.fit(all_sentences_train)

# Transform train and validation using the same vectorizer
tfidf_s1_train = tfidf.transform(data_train["sentence1_cleaned"]).toarray()
tfidf_s2_train = tfidf.transform(data_train["sentence2_cleaned"]).toarray()
tfidf_s1_val = tfidf.transform(data_val["sentence1_cleaned"]).toarray()
tfidf_s2_val = tfidf.transform(data_val["sentence2_cleaned"]).toarray()

In [ ]:
import joblib

# Create vectorizer directory
vectorizer_path = f"{artifacts_path}/{EXP_NUMBER}/vectorizer"
os.makedirs(vectorizer_path, exist_ok=True)

# Save the TF-IDF vectorizer
joblib.dump(tfidf, f"{vectorizer_path}/tfidf_vectorizer.pkl")
print(f"TF-IDF vectorizer saved to {vectorizer_path}/tfidf_vectorizer.pkl")

In [ ]:
#use the tfidf vectors as features (concatenate sentence1 and sentence2)
feature_names_s1 = [f"s1_{w}" for w in tfidf.get_feature_names_out()]
feature_names_s2 = [f"s2_{w}" for w in tfidf.get_feature_names_out()]
tf_idf_feature = feature_names_s1 + feature_names_s2

features_train = pd.DataFrame(
    np.hstack([tfidf_s1_train, tfidf_s2_train]),
    columns=tf_idf_feature
)
features_val = pd.DataFrame(
    np.hstack([tfidf_s1_val, tfidf_s2_val]),
    columns=tf_idf_feature
)

#add the features to the cleaned_data
data_tf_idf_train = pd.concat([data_train.reset_index(drop=True), features_train.reset_index(drop=True)], axis=1)
data_tf_idf_val = pd.concat([data_val.reset_index(drop=True), features_val.reset_index(drop=True)], axis=1)

#drop sentence1 and sentence2
data_tf_idf_train = data_tf_idf_train.drop(columns=['sentence1', 'sentence2', 'genre', 'annotator_labels'])
data_tf_idf_val = data_tf_idf_val.drop(columns=['sentence1', 'sentence2', 'genre', 'annotator_labels'])
data_tf_idf_train.head()

In [ ]:
len(tf_idf_feature) / 2

### feature engineering

In [ ]:
#add feature length sentence1 and sentence2
for df in (data_tf_idf_train, data_tf_idf_val):
    df['s1_length'] = df['sentence1_cleaned'].apply(lambda x: len(str(x).split()))
    df['s2_length'] = df['sentence2_cleaned'].apply(lambda x: len(str(x).split()))

In [ ]:
#add feature to detect negation words
negation_words = set(['not', 'no', 'nor', 'never', 'neither', 'none',
    'nobody', 'nothing', 'nowhere', 'noone',
    'without', 'hardly', 'scarcely', 'barely', 'seldom', 'rarely',
    'lack',
    'deny', 'refuse', 'reject', 'fail', 'avoid', 'prevent',
    'stop', 'exclude', 'oppose', 'disagree', 'neglect',
    'omit', 'miss', 'dismiss', 'discard', 'abandon',
    'cancel', 'block', 'ban', 'forbid', 'prohibit',
    'restrict', 'limit', 'decline', 'ignore',
    'contradict', 'negate', 'nullify', 'invalidate', 'revoke',
    'withdraw', 'withhold', 'suppress', 'conceal', 'hide',
    'impossible', 'unable', 'unlikely', 'insufficient', 'invalid',
    'incorrect', 'false', 'wrong', 'absent', 'missing',
    'incomplete', 'inaccurate', 'ineffective', 'inefficient',
    'unnecessary', 'unacceptable', 'unavailable', 'uncertain',
    'unclear', 'unaware', 'unfit', 'unsafe', 'unstable',
    'unsuccessful', 'unsupported', 'unwilling', 'unrelated',
    'unreliable', 'unreasonable', 'inappropriate', 'inadequate',
    'improper', 'impractical', 'improbable', 'imperfect',
    'powerless', 'helpless', 'useless', 'worthless', 'pointless',
    'hopeless', 'meaningless', 'baseless', 'groundless', 'senseless',
    'fake', 'void', 'null', 'defunct', 'obsolete',
    'failure', 'absence', 'loss', 'denial',
    'refusal', 'rejection', 'impossibility', 'prohibition',
    'restriction', 'limitation', 'contradiction', 'error',
    'mistake', 'fault', 'flaw', 'defect', 'gap',

    # ── Bigrams: auxiliary + not (from contractions) ──────────────────────────
    # spaCy splits "don't" → "do"+"not", "won't" → "will"+"not", etc.

    'can not',      # can't
    'will not',     # won't
    'do not',       # don't
    'do not',       # doesn't (does → do after lemma)
    'did not',      # didn't  (did stays as did, not lemmatized to do)
    'be not',       # isn't / aren't / wasn't / weren't  (all → be + not)
    'have not',     # haven't / hasn't (has → have after lemma)
    'have not',     # hadn't
    'would not',    # wouldn't
    'should not',   # shouldn't
    'could not',    # couldn't
    'must not',     # mustn't
    'might not',    # mightn't
    'need not',     # needn't
    'shall not',    # shan't
    'dare not',     # daren't
    'ought not',    # oughtn't
    'may not',      # mayn't

    # ── Bigrams: not + X ──────────────────────────────────────────────────────
    'not only', 'not just', 'not even', 'not yet', 'not always',
    'not really', 'not quite', 'not enough', 'not sure', 'not true',
    'not work', 'not help', 'not support', 'not allow', 'not include',
    'not have', 'not be', 'not do', 'not go', 'not make',
    'not give', 'not take', 'not know', 'not want', 'not need',
    'not say', 'not mean', 'not show', 'not seem', 'not appear',
    'not exist', 'not apply', 'not relate', 'not match', 'not fit',

    # ── Bigrams: no + X ───────────────────────────────────────────────────────
    'no long', 'no more', 'no way', 'no such', 'no need',
    'no effect', 'no result', 'no evidence', 'no proof',
    'no sign', 'no reason', 'no chance', 'no option', 'no solution',
    'no difference', 'no impact', 'no benefit', 'no use', 'no point',
    'no doubt', 'no question', 'no problem', 'no issue', 'no concern',

    # ── Bigrams: never + X ────────────────────────────────────────────────────
    'never again', 'never be', 'never have', 'never do',
    'never say', 'never show', 'never work', 'never happen',
    'never allow', 'never support',

    # ── Bigrams: other patterns ───────────────────────────────────────────────
    'neither nor',
    'by no', 'in no', 'on no',
    'far from', 'free from', 'rather than',
    'instead of', 'as oppose', 'let alone',
    'without any', 'without much', 'without enough',
    'without proper', 'without clear', 'without good',
    'fail to', 'lack of', 'deny any', 'refuse to',
    'unable to', 'hard to', 'difficult to', 'impossible to'])
    
#add feature who many negation words in sentence1 and sentence2
def count_negation_words(text):
    words = str(text).lower().split()
    return sum(1 for word in words if word in negation_words)

for df in (data_tf_idf_train, data_tf_idf_val):
    df['s1_negation_count'] = df['sentence1_cleaned'].apply(count_negation_words)
    df['s2_negation_count'] = df['sentence2_cleaned'].apply(count_negation_words)

In [ ]:
# #add feature to detect percentage of shared words 
# import nltk
# from nltk.corpus import wordnet
# from nltk.corpus import stopwords
# from nltk.stem import WordNetLemmatizer

# nltk.download('wordnet')
# nltk.download('omw-1.4')
# nltk.download('averaged_perceptron_tagger')
# nltk.download('averaged_perceptron_tagger_eng')

# lemmatizer = WordNetLemmatizer()

# def get_wordnet_pos(tag):
#     """Map POS tag to WordNet POS for better synset matching"""
#     if tag.upper().startswith('J'):
#         return wordnet.ADJ
#     elif tag.upper().startswith('V'):
#         return wordnet.VERB
#     elif tag.upper().startswith('R'):
#         return wordnet.ADV
#     else:
#         return wordnet.NOUN

# def shared_word_percentage(row):
#     s1_words = [lemmatizer.lemmatize(w.lower()) for w in str(row['sentence1_cleaned']).split()]
#     s2_words = [lemmatizer.lemmatize(w.lower()) for w in str(row['sentence2_cleaned']).split()]
    
#     #total_words = len(set(s1_words) | set(s2_words)) if len(set(s1_words) | set(s2_words)) > 0 else 1  # Avoid division by zero
    
#     synonym_count = 0
    
#     s1_tagged = nltk.pos_tag(s1_words)
#     s2_tagged = nltk.pos_tag(s2_words)
    
#     synonyms_s1 = set()
#     for word, tag in s1_tagged:
#         pos = get_wordnet_pos(tag)
#         for syn in wordnet.synsets(word, pos=pos):
#             for lemma in syn.lemmas():
#                 synonyms_s1.add(lemmatizer.lemmatize(lemma.name().lower()))
                
#     for word, tag in s2_tagged:
#         pos = get_wordnet_pos(tag)
#         synonyms_word_s2 = set()
#         for syn in wordnet.synsets(word, pos=pos):
#             for lemma in syn.lemmas():
#                 synonyms_word_s2.add(lemmatizer.lemmatize(lemma.name().lower()))
        
#         if synonyms_word_s2 & synonyms_s1:
#             synonym_count += 1 
        
#     return synonym_count

# for df in (data_tf_idf_train, data_tf_idf_val):
#     df['count_word_shared'] = df.apply(shared_word_percentage, axis=1)

def shared_word_percentage(row):
    s1_words = set(str(row['sentence1_cleaned']).split())
    s2_words = set(str(row['sentence2_cleaned']).split())
    if len(s1_words) == 0 or len(s2_words) == 0:
        return 0.0
    shared_words = s1_words.intersection(s2_words)
    total_words = s1_words.union(s2_words)
    return len(shared_words) / len(total_words) if len(total_words) > 0 else 0.0

for df in (data_tf_idf_train, data_tf_idf_val):
    df['ratio_word_shared_sentence1_sentence2'] = df.apply(shared_word_percentage, axis=1)

In [ ]:
# # add feature to detect antonyms between sentence1 and sentence2
# import nltk
# from nltk.corpus import wordnet
# from nltk.stem import WordNetLemmatizer

# nltk.download('wordnet')
# nltk.download('omw-1.4')

# def get_wordnet_pos(tag):
#     """Map POS tag to WordNet POS for better synset matching"""
#     if tag.upper().startswith('J'):
#         return wordnet.ADJ
#     elif tag.upper().startswith('V'):
#         return wordnet.VERB
#     elif tag.upper().startswith('R'):
#         return wordnet.ADV
#     else:
#         return wordnet.NOUN

# lemmatizer = WordNetLemmatizer()

# def antonym_word_percentage(row):
#     s1_words = [lemmatizer.lemmatize(w.lower()) for w in str(row['sentence1_cleaned']).split()]
#     s2_words = [lemmatizer.lemmatize(w.lower()) for w in str(row['sentence2_cleaned']).split()]
    
#     total_words = len(set(s1_words) | set(s2_words)) if len(set(s1_words) | set(s2_words)) > 0 else 1  # Avoid division by zero
    
#     antonym_count = 0
    
#     s1_tagged = nltk.pos_tag(s1_words)
#     s2_tagged = nltk.pos_tag(s2_words)
    
#     antonyms_s1 = set()
#     for word, tag in s1_tagged:
#         pos = get_wordnet_pos(tag)
#         for syn in wordnet.synsets(word, pos=pos):
#             for lemma in syn.lemmas():
#                 for ant in lemma.antonyms():
#                     antonyms_s1.add(lemmatizer.lemmatize(ant.name().lower()))
                
#     for word, tag in s2_tagged:
#         pos = get_wordnet_pos(tag)
#         synonyms_word_s2 = set()
#         for syn in wordnet.synsets(word, pos=pos):
#             for lemma in syn.lemmas():
#                 synonyms_word_s2.add(lemmatizer.lemmatize(lemma.name().lower()))
        
#         if synonyms_word_s2 & antonyms_s1:
#             antonym_count += 1 
        
#     return antonym_count / total_words

# for df in (data_tf_idf_train, data_tf_idf_val):
#     df['ratio_antonym_sentence1_sentence2'] = df.apply(antonym_word_percentage, axis=1)

# add feature to detect antonyms between sentence1 and sentence2
import nltk
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer

nltk.download('wordnet')
nltk.download('omw-1.4')

lemmatizer = WordNetLemmatizer()

def count_antonyms(row):
    s1_words = [lemmatizer.lemmatize(w.lower()) for w in str(row['sentence1_cleaned']).split()]
    s2_words = [lemmatizer.lemmatize(w.lower()) for w in str(row['sentence2_cleaned']).split()]
    
    total_words = len(set(s1_words) | set(s2_words)) if len(set(s1_words) | set(s2_words)) > 0 else 1  # Avoid division by zero
    
    antonym_count = 0
    
    for word in s1_words:
        antonyms = set()
        for syn in wordnet.synsets(word):
            for lemma in syn.lemmas():
                for ant in lemma.antonyms():  # include all antonyms
                    antonyms.add(lemmatizer.lemmatize(ant.name().lower()))
        
        # Increment if any antonym exists in S2
        if any(ant in s2_words for ant in antonyms):
            antonym_count += 1
            
    return antonym_count / total_words

for df in (data_tf_idf_train, data_tf_idf_val):
    df['ratio_antonym_sentence1_sentence2'] = df.apply(count_antonyms, axis=1)

**feature combinaison**

In [ ]:
for df in (data_tf_idf_train, data_tf_idf_val):
    df["ratio_length_sentence1_sentence2"] = df["s1_length"] / (df["s2_length"] + 1) 
    df['ratio_negation_count_sentence1_sentence2'] = df['s1_negation_count'] / (df['s2_negation_count'] + 1)

**keep only required features**

In [ ]:
#keep only feature needed for training
added_feature = ["ratio_word_shared_sentence1_sentence2",
                                    "ratio_length_sentence1_sentence2",
                                    "ratio_negation_count_sentence1_sentence2",
                                    "ratio_antonym_sentence1_sentence2",
                                    "s2_negation_count",
                                    "s2_length"
                                ]
features_to_keep = tf_idf_feature + added_feature + ["gold_label"]

data_tf_idf_train = data_tf_idf_train[features_to_keep]
data_tf_idf_val = data_tf_idf_val[features_to_keep]

In [ ]:
# Plot average of all features by gold_label
import matplotlib.pyplot as plt
import seaborn as sns

# Calculate mean features per gold_label (exclude gold_label itself)
avg_features = data_tf_idf_train.groupby('gold_label')[added_feature].mean()

# Plot
fig, axes = plt.subplots(figsize=(14, 8))
avg_features.T.plot(kind='bar', ax=axes)
plt.title('Average Feature Values by Gold Label', fontsize=14, fontweight='bold')
plt.xlabel('Features', fontsize=12)
plt.ylabel('Average Value', fontsize=12)
plt.legend(title='Gold Label', labels=['Contradiction', 'Entailment', 'Neutral'])
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(f"Average feature values by gold_label:\n{avg_features}")

### train models

In [ ]:
#use the pre-split data for train and validation

#drop columns gold_label
cleaned_data_train = data_tf_idf_train.drop(columns=['gold_label'])
cleaned_data_val = data_tf_idf_val.drop(columns=['gold_label'])

X_train, X_test = cleaned_data_train, cleaned_data_val
y_train, y_test = data_tf_idf_train['gold_label'], data_tf_idf_val['gold_label']

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
import xgboost as xgb
from sklearn.svm import SVC
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import json

models = {
    # "Naive Bayes": GaussianNB(),
    # "KNeighbor Classifier": KNeighborsClassifier(),
    "XGBoost": xgb.XGBClassifier(max_depth=8,
        learning_rate=0.19645372835376285,
        n_estimators=176,
        subsample=0.9825627470813624,
        colsample_bytree=0.9113205212356545,
        min_child_weight=7,
        gamma=1.8034493483949205),
    #"Random Forest": RandomForestClassifier(n_jobs=-1),
    # "Logistic Regression": LogisticRegression(multi_class='multinomial', n_jobs=-1),
    # "SVM": SVC(kernel='linear', decision_function_shape='ovo')
}

# encode labels for XGBoost (needs integer classes)
label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)

for model_name, model in models.items():
    print(f"Training {model_name}...")

    if model_name == "XGBoost":
        model.fit(X_train, y_train_enc)
        pred_enc = model.predict(X_test)
        pred = label_encoder.inverse_transform(pred_enc)
        class_names = label_encoder.classes_
    # elif model_name in {'SVM', 'Logistic Regression', 'KNeighbor Classifier'}:
    #     model.fit(X_train_svd, y_train)
    #     pred = model.predict(X_test_svd)
    #     class_names = model.classes_
    # elif model_name in {'Naive Bayes', 'Random Forest'}:
    #     model.fit(X_train, y_train)
    #     pred = model.predict(X_test)
    #     class_names = model.classes_

    # --- per-class metrics as dict ---
    per_class_metrics = {}
    for cls in class_names:
        per_class_metrics[cls] = {
            "precision": float(precision_score(y_test, pred, average=None, labels=[cls])[0]),
            "recall": float(recall_score(y_test, pred, average=None, labels=[cls])[0]),
            "f1": float(f1_score(y_test, pred, average=None, labels=[cls])[0])
        }

    # --- overall metrics ---
    accuracy = accuracy_score(y_test, pred)
    precision_weighted = precision_score(y_test, pred, average='weighted')
    recall_weighted = recall_score(y_test, pred, average='weighted')
    f1_weighted = f1_score(y_test, pred, average='weighted')

    # save metrics json
    metrics = {
        "per_class_metrics": per_class_metrics,
        "accuracy": accuracy,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted
    }
    
    #confusion matrix
    cm = confusion_matrix(y_test, pred, labels=class_names)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title(f'Confusion Matrix for {model_name}')
    plt.savefig(f"../exp_results/artifacts/{EXP_NUMBER}/confusion_matrix.png")
    plt.close()
    

    with open(f"../exp_results/artifacts/{EXP_NUMBER}/metrics.json", "w") as f:
        json.dump(metrics, f, indent=4)
    
    # Save the trained model
    model_path = f"../exp_results/artifacts/{EXP_NUMBER}/model"
    os.makedirs(model_path, exist_ok=True)
    joblib.dump(model, f"{model_path}/model.pkl")
    print(f"Model saved to {model_path}/model.pkl")
    
    # Save label encoder if XGBoost
    if model_name == "XGBoost":
        joblib.dump(label_encoder, f"{model_path}/label_encoder.pkl")
        print(f"Label encoder saved to {model_path}/label_encoder.pkl")